# Restaurant Clustering

Builds the clustering outputs used by the dashboard:
- `data_output/reviews.csv`: review-level text corpus with cluster labels
- `data_output/clustering_results.csv`: restaurant-level cluster assignments with 2D coordinates


In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from umap import UMAP

BASE_DIR = Path("..")
KOL_PATH = BASE_DIR / "data" / "kol" / "KOL_Posts.csv"
PLACES_PATH = BASE_DIR / "_1_eda" / "data_output" / "places_api_new_results.csv"
REVIEWS_PATH = BASE_DIR / "_3_marketing" / "data_output" / "restaurant_reviews.parquet"
OUTPUT_DIR = Path("data_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

kol_posts = pd.read_csv(KOL_PATH)
places = pd.read_csv(PLACES_PATH)
restaurant_reviews = pd.read_parquet(REVIEWS_PATH)


## 1. Build the Restaurant Text Corpus


In [ ]:
kol_aggregated = (
    kol_posts.groupby("Restaurant Name")["Content"]
    .apply(lambda s: " ".join(s.dropna().astype(str)))
    .reset_index(name="kol_text")
)

places = places.copy()
places["google_text"] = (
    places["input_string"].fillna("").astype(str)
    + " "
    + places["raw_types"].fillna("").astype(str).str.replace(",", " ")
    + " "
    + places["Cuisine"].fillna("").astype(str)
)

place_text = (
    places[["input_string", "google_text"]]
    .rename(columns={"input_string": "restaurant name"})
    .drop_duplicates("restaurant name")
)

review_text = (
    restaurant_reviews.groupby("input_restaurant_name")["review_text"]
    .apply(lambda s: " ".join(s.dropna().astype(str)))
    .reset_index(name="review_text")
    .rename(columns={"input_restaurant_name": "restaurant name"})
)

reviews = (
    review_text
    .merge(place_text, on="restaurant name", how="left")
    .merge(kol_aggregated.rename(columns={"Restaurant Name": "restaurant name"}), on="restaurant name", how="left")
)
reviews["raw_text"] = (
    reviews["review_text"].fillna("").astype(str)
    + " "
    + reviews["google_text"].fillna("").astype(str)
    + " "
    + reviews["kol_text"].fillna("").astype(str)
).str.replace(r"\s+", " ", regex=True).str.strip()

reviews = reviews[reviews["raw_text"].str.len() > 0].copy()
reviews.head()


## 2. Clean and Vectorize Text


In [ ]:
redundant_words = [
    "point_of_interest",
    "establishment",
    "general",
    "food establishment",
    "restaurant point_of_interest",
    "food point_of_interest",
    "point_of_interest establishment",
    "establishment general",
    "restaurant food",
    "meal_delivery",
    "meal_takeaway",
    "lodging",
    "tourist_attraction",
    "night_club",
    "shopping_mall",
    "bangkok",
    "sukhumvit",
    "singapore",
    "thai",
    "baht",
    "order",
    "ordered",
    "dishes",
    "fried",
    "sauce",
    "spicy",
    "time",
    "eat",
    "meat",
    "restaurant",
    "soup",
    "super",
    "really",
    "terrible",
    "floor",
    "rice",
    "pork",
    "beef",
    "chicken",
    "crab",
]

def clean_text(text: str) -> str:
    cleaned = str(text).lower()
    for phrase in sorted(redundant_words, key=len, reverse=True):
        cleaned = cleaned.replace(phrase, " ")
    cleaned = re.sub(r"[^a-z\s]", " ", cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned

reviews["clean_text"] = reviews["raw_text"].apply(clean_text)
reviews = reviews[reviews["clean_text"].str.len() > 0].copy()

vectorizer = TfidfVectorizer(
    max_features=200,
    ngram_range=(1, 2),
    stop_words="english",
    min_df=2,
)
text_vectors = vectorizer.fit_transform(reviews["clean_text"])
print(reviews.shape)


## 3. Cluster Reviews


In [ ]:
cluster_themes = {
    0: "Bar & Alcoholic Drinks",
    1: "Good food and atmosphere",
    2: "Buffet & premium dining",
    3: "Excellent service",
    4: "Cafes, Coffee, Breakfast",
}

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
reviews["cluster"] = kmeans.fit_predict(text_vectors)
reviews["theme"] = reviews["cluster"].map(cluster_themes)
reviews.to_csv(OUTPUT_DIR / "reviews.csv", index=False)

reviews[["restaurant name", "cluster", "theme"]].head()


## 4. Build Restaurant-Level Assignments


In [ ]:
reducer = UMAP(
    n_components=2,
    n_neighbors=30,
    min_dist=1.0,
    metric="cosine",
    random_state=42,
)
embeddings_2d = reducer.fit_transform(text_vectors.toarray())

review_points = reviews.copy()
review_points["x"] = embeddings_2d[:, 0]
review_points["y"] = embeddings_2d[:, 1]

def dominant_value(series: pd.Series):
    counts = series.value_counts()
    return counts.index[0] if len(counts) else pd.NA

def dominant_share(series: pd.Series) -> float:
    counts = series.value_counts(normalize=True)
    return float(counts.iloc[0]) if len(counts) else np.nan

clustering_results = (
    review_points.groupby("restaurant name", as_index=False)
    .agg(
        cluster=("cluster", dominant_value),
        theme=("theme", dominant_value),
        x=("x", "mean"),
        y=("y", "mean"),
        cluster_confidence=("cluster", dominant_share),
        review_count=("raw_text", "size"),
    )
    .sort_values(["cluster", "restaurant name"])
    .reset_index(drop=True)
)

clustering_results.to_csv(OUTPUT_DIR / "clustering_results.csv", index=False)
clustering_results.head()
